In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path
from scipy.spatial import ConvexHull
from pymatgen.core.composition import Composition

# Plot style
plt.rcParams.update({
    "font.size": 15,
    "axes.titlesize": 20,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "legend.title_fontsize": 15,
})

# EXCLUDE_ORDERINGS = {"JLK_c"}
EXCLUDE_ORDERINGS = {}

def energy_per_TMO2(row):
    """Return energy per TMO2 unit in eV."""
    formula = row["formula"]
    tm_symbol = row["tm"]
    natoms = row["natoms"]

    comp_norm = Composition(formula)
    n_norm = comp_norm.num_atoms  # e.g. 22 for Na(CoO2)7

    if n_norm == 0 or natoms == 0:
        return np.nan

    scale = natoms / n_norm
    n_TMO2_units = comp_norm[tm_symbol] * scale

    if n_TMO2_units == 0:
        return np.nan

    if "energy_eV" in row and not pd.isna(row["energy_eV"]):
        E_tot = row["energy_eV"]
    elif "energy_per_atom_eV" in row and not pd.isna(row["energy_per_atom_eV"]):
        E_tot = row["energy_per_atom_eV"] * natoms
    else:
        return np.nan

    return E_tot / n_TMO2_units


def lower_hull_indices_scipy(xs, ys):
    """
    Compute indices of points on the lower convex hull (in energy)
    using scipy.spatial.ConvexHull.

    - xs, ys: 1D arrays of the same length
    - Returns indices into the original arrays
    """
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    n = len(xs)
    if n == 0:
        return []
    # Collapse multiple points at the same x to the lowest-energy one
    unique_indices = []
    for x0 in sorted(set(xs)):
        maskx = np.isclose(xs, x0)
        idxs = np.where(maskx)[0]
        j = idxs[np.argmin(ys[idxs])]
        unique_indices.append(j)

    xs_u = xs[unique_indices]
    ys_u = ys[unique_indices]

    if len(xs_u) <= 2:
        # Everything is on the "hull"
        return sorted(unique_indices, key=lambda i: xs[i])

    pts = np.column_stack([xs_u, ys_u])
    hull = ConvexHull(pts)
    verts = hull.vertices  # indices into xs_u / ys_u

    # Sort hull vertices by x, then y
    verts_sorted = sorted(verts, key=lambda i: (xs_u[i], ys_u[i]))

    # One pass to keep only the lower chain
    lower_u = []
    for i in verts_sorted:
        while len(lower_u) >= 2:
            j = lower_u[-2]
            k = lower_u[-1]
            x1, y1 = xs_u[j], ys_u[j]
            x2, y2 = xs_u[k], ys_u[k]
            x3, y3 = xs_u[i], ys_u[i]
            cross = (x2 - x1) * (y3 - y1) - (y2 - y1) * (x3 - x1)
            if cross <= 0:
                # New point is below or collinear in an "upward" sense:
                # remove the previous point from the lower hull
                lower_u.pop()
            else:
                break
        lower_u.append(i)

    # Map back to original indices
    lower_idx = [unique_indices[i] for i in lower_u]
    return sorted(lower_idx, key=lambda i: xs[i])


def short_name(ordering: str) -> str:
    return ordering.replace("_w_vac", "")


# Load DFT summary (from dft_collect_from_single_TM.py)
# df = pd.read_csv("../data/singleTM_w_exp_2nd_relax.csv")  # adjust path as needed
df = pd.read_csv("../data/singleTM_tighter_conv.csv")

# Keep only clean, converged runs.
mask = pd.Series(True, index=df.index)
if "status" in df.columns:
    mask &= df["status"] == "OK"
if "n_custodian_errors" in df.columns:
    mask &= df["n_custodian_errors"] == 0
if "converged" in df.columns:
    mask &= df["converged"] == True

df = df[mask].copy()
df["E_TMO2_eV"] = df.apply(energy_per_TMO2, axis=1)
df = df.dropna(subset=["E_TMO2_eV"]).copy()

# Map ordering -> x_Na by hard coded map.
na_map = {
    "AJT_a": 1/7, "AJT_b": 1/4, "AJT_c": 1/4, "AJT_d": 1/3, "AJT_e": 1/3, 
    "AJT_f": 2/5, "AJT_g": 1/2, "AJT_h": 3/5, "AJT_i": 5/8, "AJT_j": 5/8,
    "AJT_k": 2/3, "AJT_l": 2/3, "AJT_m": 2/3, "AJT_n": 3/4, "AJT_o": 3/4,
    "AJT_p": 5/6, "AJT_q": 7/8, "AJT_r": 8/9,
    "JLK_b": 3/8, "JLK_c": 7/10, "JLK_theta_4": 4/5, "JLK_theta_5": 5/6,
    "JLK_delta": 2/3, "JLK_eta": 4/7, "JLK_zeta": 1/2, "JLK_zeta2": 2/5,
    "JLK_zeta3": 3/7, "JLK_zeta6": 6/11,
    "CD_a": 1/2, "JMAS_a": 1/3, "JMAS_b": 2/5, "JMAS_e": 1, "XC_a": 1/3,
    "XC_b": 1/2, "JMAS_c": 1/2, "JMAS_d": 2/3, "XL_a": 5/8
}

# Prefer meta_ordering if available, else fallback to ordering
ordering_raw = df["meta_ordering"].fillna(df["ordering"])
df["ordering_short"] = ordering_raw.apply(short_name)
if EXCLUDE_ORDERINGS:
    df = df[~df["ordering_short"].isin(EXCLUDE_ORDERINGS)].copy()

# First map all the enumerated orderings
df["xNa"] = df["ordering_short"].map(na_map)

# Override for endmembers:
mask_full = df["ordering"] == "full"
mask_empty = df["ordering"] == "empty"
df.loc[mask_full, "xNa"] = 1.0
df.loc[mask_empty, "xNa"] = 0.0

# Drop anything that still doesn't have an xNa
df = df.dropna(subset=["xNa"]).copy()

# Label with the chosen group (from metadata if available)
if "meta_group_min" in df.columns:
    df["group_label"] = df["meta_group_min"].fillna("")
else:
    df["group_label"] = ""

# Plotting convex hulls per TM
tms = sorted(df["tm"].unique())
out_paths = []

for tm in tms:
    df_tm = df[df["tm"] == tm].copy()
    if df_tm.empty:
        continue

    # One point per (ordering_short, phase, xNa)
    df_pts = (
        df_tm
        .groupby(["ordering_short", "phase", "xNa"], as_index=False)
        .agg({"E_TMO2_eV": "mean", "group_label": "first"})
    )

    if df_pts.empty:
        continue

    xs = df_pts["xNa"].to_numpy()
    ys = df_pts["E_TMO2_eV"].to_numpy()

    # Convex hull indices on *raw* energies (affine shift won't change hull)
    hull_idx = lower_hull_indices_scipy(xs, ys)
    hull_mask = np.zeros(len(df_pts), dtype=bool)
    hull_mask[hull_idx] = True

    # --- Subtract tie-line between hull endpoints so they sit at ΔE = 0 ---
    if len(hull_idx) >= 2:
        iL, iR = hull_idx[0], hull_idx[-1]
        xL, yL = xs[iL], ys[iL]
        xR, yR = xs[iR], ys[iR]

        if xR != xL:
            y_ref = yL + (yR - yL) * (xs - xL) / (xR - xL)
            ys_rel = ys - y_ref
        else:
            ys_rel = ys - ys.min()
    else:
        ys_rel = ys - ys.min()

    df_pts["E_rel_TMO2_eV"] = ys_rel

    fig, ax = plt.subplots(figsize=(6, 4.5))

    # Phase masks
    is_o3 = df_pts["phase"].str.contains("O3")
    is_p3 = df_pts["phase"].str.contains("P3")
    is_o1 = df_pts["phase"].str.contains("O1")
    mask_hull = hull_mask
    mask_non = ~hull_mask

    # --- Non-hull points (red outline) ---
    mask = is_o3 & mask_non
    if mask.any():
        ax.scatter(
            df_pts.loc[mask, "xNa"], df_pts.loc[mask, "E_rel_TMO2_eV"],
            marker="o", s=130, facecolors="none",
            edgecolors="red", linewidths=1.7,
        )

    mask = is_p3 & mask_non
    if mask.any():
        ax.scatter(
            df_pts.loc[mask, "xNa"], df_pts.loc[mask, "E_rel_TMO2_eV"],
            marker="s", s=130, facecolors="none",
            edgecolors="red", linewidths=1.7,
        )

    mask = is_o1 & mask_non
    if mask.any():
        ax.scatter(
            df_pts.loc[mask, "xNa"], df_pts.loc[mask, "E_rel_TMO2_eV"],
            marker="^", s=130, facecolors="none",
            edgecolors="red", linewidths=1.7,
        )

    # --- Hull points (black outline) ---
    mask = is_o3 & mask_hull
    if mask.any():
        ax.scatter(
            df_pts.loc[mask, "xNa"], df_pts.loc[mask, "E_rel_TMO2_eV"],
            marker="o", s=170, facecolors="none",
            edgecolors="black", linewidths=2.2,
        )

    mask = is_p3 & mask_hull
    if mask.any():
        ax.scatter(
            df_pts.loc[mask, "xNa"], df_pts.loc[mask, "E_rel_TMO2_eV"],
            marker="s", s=170, facecolors="none",
            edgecolors="black", linewidths=2.2,
        )

    mask = is_o1 & mask_hull
    if mask.any():
        ax.scatter(
            df_pts.loc[mask, "xNa"], df_pts.loc[mask, "E_rel_TMO2_eV"],
            marker="^", s=170, facecolors="none",
            edgecolors="black", linewidths=2.2,
        )

    # Hull connecting line (on relative energies)
    if len(hull_idx) > 1:
        xs_arr = df_pts["xNa"].to_numpy()
        e_rel_arr = df_pts["E_rel_TMO2_eV"].to_numpy()
        hx = [xs_arr[i] for i in hull_idx]
        hy = [e_rel_arr[i] for i in hull_idx]
        ax.plot(hx, hy, color="black", linewidth=2)

        # Annotate hull points with ordering only
        y_span = max(hy) - min(hy) if len(hy) > 1 else 0.01
        y_offset = 0.03 * y_span
        for i in hull_idx:
            row_pt = df_pts.iloc[i]
            label = str(row_pt["ordering_short"])
            ax.text(
                xs_arr[i] - 0.05,
                e_rel_arr[i] - y_offset * 4.4,
                label,
                fontsize=11,
                ha="left",
                va="bottom",
                rotation=0,
            )

    # ---- Custom legend: shapes for phases, colors for hull status ----
    phase_handles = [
        Line2D(
            [], [], marker="o", linestyle="None",
            markerfacecolor="none", markeredgecolor="black",
            markersize=10, label="O3",
        ),
        Line2D(
            [], [], marker="s", linestyle="None",
            markerfacecolor="none", markeredgecolor="black",
            markersize=10, label="P3",
        ),
        Line2D(
            [], [], marker="^", linestyle="None",
            markerfacecolor="none", markeredgecolor="black",
            markersize=10, label="O1",
        ),
    ]

    status_handles = [
        Line2D(
            [], [], marker="o", linestyle="None",
            markerfacecolor="none", markeredgecolor="black",
            markersize=10, label="hull (black)",
        ),
        Line2D(
            [], [], marker="o", linestyle="None",
            markerfacecolor="none", markeredgecolor="red",
            markersize=10, label="non-hull (red)",
        ),
    ]

    handles = phase_handles + status_handles

    # Legend outside the axes on the right
    ax.legend(
        handles=handles,
        frameon=False,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        borderaxespad=0.0,
    )


    ax.set_xlabel(rf"x in Na$_{{\mathrm{{x}}}}${tm}O$_2$")
    ax.set_ylabel(f"Formation Energy (eV/{tm}O$_2$)")
    # ax.set_title(f"Convex Hull: TM = {tm}")

    # Slight padding in x and y
    ax.set_xlim(-0.02, 1.02)
    y_vals = df_pts["E_rel_TMO2_eV"].to_numpy()
    y_min = float(np.min(y_vals))
    # small margin below minimum, but cap at 50 meV on top
    y_margin = 0.05 * (1.0 - y_min if 1.0 > y_min else 1.0)
    ax.set_ylim(y_min - y_margin, 0.1)

    ax.grid(True, linestyle="--", alpha=0.2)
    ax.set_axisbelow(True)

    outpath = Path(f"../data/figures/singleTM/convexhull_tighterconv_{tm}.pdf")
    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    out_paths.append(str(outpath))

out_paths


['../data/figures/singleTM/convexhull_tighterconv_Co.pdf',
 '../data/figures/singleTM/convexhull_tighterconv_Cr.pdf',
 '../data/figures/singleTM/convexhull_tighterconv_Fe.pdf',
 '../data/figures/singleTM/convexhull_tighterconv_Mn.pdf',
 '../data/figures/singleTM/convexhull_tighterconv_Ni.pdf',
 '../data/figures/singleTM/convexhull_tighterconv_Ti.pdf',
 '../data/figures/singleTM/convexhull_tighterconv_V.pdf']